In [1]:
# ==========================
# 1. Bibliotecas e parâmetros
# ==========================
import platform
import sqlite3
from pathlib import Path

import pandas as pd

ANO = 2026
DATA_INICIO = f"{ANO}-01-01"
TABELA_TARIFAS = f"Danone_Tarifas_{ANO}"
TABELA_CLIENTES = f"danone_clientes_{ANO}"
RECRIAR_TABELAS = True


In [3]:
# ==========================
# 2. Caminhos
# ==========================
if platform.system() == "Windows":
    DB_PATH = Path(
        r"C:\Users\LISARR\Documents\python\01.Financeiro\inform_27.db"
    )
    PASTA_DADOS = Path(
        r"C:\Users\LISARR\Documents\python\01.Financeiro\2026_dados"
    )

elif platform.system() == "Darwin":
    DB_PATH = Path(
        "/Users/rr/Library/Mobile Documents/"
        "com~apple~CloudDocs/05.Salvesen/inform_27.db"
    )
    PASTA_DADOS = Path(
        "/Users/rr/Library/Mobile Documents/"
        "com~apple~CloudDocs/05.Salvesen"
    )

else:
    DB_PATH = Path("inform_27.db")
    PASTA_DADOS = Path(".")

padroes = (
    "Danone_Custos_v2.xlsx",
    "Danone_Custos_v2*.xlsx",
)

ficheiros_danone = []

for padrao in padroes:
    ficheiros_danone.extend(
        caminho
        for caminho in PASTA_DADOS.rglob(padrao)
        if caminho.is_file() and not caminho.name.startswith("~$")
    )

ficheiros_danone = sorted(set(ficheiros_danone))

if not ficheiros_danone:
    raise FileNotFoundError(
        f"Ficheiro Danone_Custos_v2.xlsx não encontrado em: {PASTA_DADOS}"
    )

FICHEIRO_DANONE = max(
    ficheiros_danone,
    key=lambda caminho: caminho.stat().st_size,
)
DB_PATH.parent.mkdir(parents=True, exist_ok=True)

print(f"✓ BD: {DB_PATH}")
print(f"✓ Danone: {FICHEIRO_DANONE}")


✓ BD: /Users/rr/Library/Mobile Documents/com~apple~CloudDocs/05.Salvesen/inform_27.db
✓ Danone: /Users/rr/Library/Mobile Documents/com~apple~CloudDocs/05.Salvesen/Danone_Custos_v2.xlsx


In [4]:
# ==========================
# 3. Tarifas-base de 2026
# ==========================
TARIFAS_EUR_TON = {
    "Alentejo": (157.465344, "TRANSPORTE"),
    "Algarve": (81.357095, "TRANSPORTE"),
    "Azambuja 1": (14.434323, "TRANSPORTE"),
    "Azambuja 2": (27.556435, "TRANSPORTE"),
    "Azambuja 3": (52.488448, "TRANSPORTE"),
    "Centro": (38.054125, "TRANSPORTE"),
    "Export": (52.488448, "TRANSPORTE"),
    "EXPORT": (52.488448, "TRANSPORTE"),
    "GS Centro": (94.479207, "TRANSPORTE"),
    "GS Norte": (141.718810, "TRANSPORTE"),
    "LEIRIA ( Wholesaler)": (123.735144, "TRANSPORTE"),
    "Norte 1": (31.493069, "TRANSPORTE"),
    "Norte 2": (38.054125, "TRANSPORTE"),
    "Norte 3": (78.732672, "TRANSPORTE"),
    "VALUE NETWORKS": (93.543769, "TRANSPORTE"),
    "Wholesalers Norte": (107.732735, "TRANSPORTE"),
    "TGTG": (37.677351, "TRANSPORTE"),
    "FILE HOTEIS LDA": (140.315653, "TRANSPORTE"),
    "Directos Lisboa": (91.852620, "Directos tte"),
    "Directos Oporto": (167.699280, "Directos tte"),
    "Directos Coimbra": (167.699280, "Directos tte"),
    "Porto Prevenda": (31.960000, "Transporte Primario"),
    "PV Algarve": (81.350000, "Transporte Primario"),
}

TIPOS_CAPILAR = {
    "Capilar",
    "PV Coimbra",
    "PV Coina",
    "PV Leiria",
    "PV Lisboa",
    "PV Porto",
}

TIPOS_SEM_INGRESSO = {
    "AUCHAN sin Tte.",
    "Recolha Armazém",
    "S/ Valor de Tarifa (Recolha em Cais)",
    "Sin Transporte Marketing",
    "Sin Transporte pérdida",
}

TIPOS_POR_DEFINIR = {
    "NO",
    "Pendiente",
    "SEM_MAPEAMENTO",
}

registos_tarifas = [
    (
        tipo_local,
        "POR_TONELADA",
        "EUR_TON",
        tarifa_base,
        DATA_INICIO,
        None,
        fonte_tarifa,
    )
    for tipo_local, (tarifa_base, fonte_tarifa)
    in TARIFAS_EUR_TON.items()
]

registos_tarifas.append(
    (
        "Ofertas",
        "POR_ENTREGA",
        "EUR_ENTREGA",
        28.830290,
        DATA_INICIO,
        None,
        "Ofertas",
    )
)

registos_tarifas.extend(
    (
        tipo_local,
        "CAPILAR",
        None,
        None,
        DATA_INICIO,
        None,
        "Capilar",
    )
    for tipo_local in sorted(TIPOS_CAPILAR)
)

registos_tarifas.extend(
    (
        tipo_local,
        "SEM_INGRESSO",
        "ZERO",
        0.0,
        DATA_INICIO,
        None,
        "Sem transporte",
    )
    for tipo_local in sorted(TIPOS_SEM_INGRESSO)
)

registos_tarifas.extend(
    (
        tipo_local,
        "POR_DEFINIR",
        None,
        None,
        DATA_INICIO,
        None,
        "Por definir",
    )
    for tipo_local in sorted(TIPOS_POR_DEFINIR)
)

tarifas = pd.DataFrame(
    registos_tarifas,
    columns=[
        "tipo_local",
        "modelo_ingresso",
        "unidade",
        "tarifa_base",
        "data_inicio",
        "data_fim",
        "fonte_tarifa",
    ],
).sort_values("tipo_local").reset_index(drop=True)

tarifas


,tipo_local,modelo_ingresso,unidade,tarifa_base,data_inicio,data_fim,fonte_tarifa
0,AUCHAN sin Tte.,SEM_INGRESSO,ZERO,0.000000,2026-01-01,None,Sem transporte
1,Alentejo,POR_TONELADA,EUR_TON,157.465344,2026-01-01,None,TRANSPORTE
2,Algarve,POR_TONELADA,EUR_TON,81.357095,2026-01-01,None,TRANSPORTE
3,Azambuja 1,POR_TONELADA,EUR_TON,14.434323,2026-01-01,None,TRANSPORTE
4,Azambuja 2,POR_TONELADA,EUR_TON,27.556435,2026-01-01,None,TRANSPORTE
5,Azambuja 3,POR_TONELADA,EUR_TON,52.488448,2026-01-01,None,TRANSPORTE
6,Capilar,CAPILAR,None,NaN,2026-01-01,None,Capilar
7,Centro,POR_TONELADA,EUR_TON,38.054125,2026-01-01,None,TRANSPORTE
8,Directos Coimbra,POR_TONELADA,EUR_TON,167.699280,2026-01-01,None,Directos tte
9,Directos Lisboa,POR_TONELADA,EUR_TON,91.852620,2026-01-01,None,Directos tte


In [5]:
# ==========================
# 4. Mapeamento dos clientes
# ==========================
df = pd.read_excel(
    FICHEIRO_DANONE,
    sheet_name="KILOSPO",
    usecols=[
        "PFEENT",
        "PCODCL",
        "PNOMCL",
        "Ruta WH",
        "Ruta Tte. Nueva",
    ],
)

df.columns = df.columns.astype(str).str.strip()

colunas_texto = [
    "PCODCL",
    "PNOMCL",
    "Ruta WH",
    "Ruta Tte. Nueva",
]

for coluna in colunas_texto:
    df[coluna] = df[coluna].astype("string").str.strip()

df["PCODCL"] = df["PCODCL"].str.replace(
    r"\.0$",
    "",
    regex=True,
)

df["data_entrega"] = pd.to_datetime(
    df["PFEENT"].astype("Int64").astype("string"),
    format="%Y%m%d",
    errors="coerce",
)

df = df[df["data_entrega"].dt.year.eq(ANO)].copy()
df = df.dropna(subset=["PCODCL"]).copy()
df["Ruta Tte. Nueva"] = df["Ruta Tte. Nueva"].fillna(
    "SEM_MAPEAMENTO"
)

conflitos = (
    df.groupby("PCODCL")["Ruta Tte. Nueva"]
    .nunique()
    .loc[lambda serie: serie > 1]
)

clientes = (
    df.sort_values(["PCODCL", "data_entrega"])
    .groupby("PCODCL", as_index=False)
    .tail(1)
    [["PCODCL", "PNOMCL", "Ruta WH", "Ruta Tte. Nueva"]]
    .sort_values("PCODCL")
    .reset_index(drop=True)
)

tipos_com_regra = set(tarifas["tipo_local"])
tipos_sem_regra = sorted(
    set(clientes["Ruta Tte. Nueva"].dropna()) - tipos_com_regra
)

if tipos_sem_regra:
    raise ValueError(
        f"Tipos locais sem regra de tarifa: {tipos_sem_regra}"
    )

if not conflitos.empty:
    display(
        df[df["PCODCL"].isin(conflitos.index)]
        .sort_values(["PCODCL", "data_entrega"])
        [[
            "PCODCL",
            "PNOMCL",
            "Ruta WH",
            "Ruta Tte. Nueva",
            "data_entrega",
        ]]
        .drop_duplicates()
    )

print(f"✓ Clientes preparados: {len(clientes):,}")
print(f"✓ Tarifas preparadas: {len(tarifas):,}")

clientes


✓ Clientes preparados: 1,047
✓ Tarifas preparadas: 38


,PCODCL,PNOMCL,Ruta WH,Ruta Tte. Nueva
0,350135956,GOUVEIA E MENDES,PV Lisboa,PV Lisboa
1,350136151,"NAVARRAS SUPERMERCADOS, LDA",Porto Prevenda,Porto Prevenda
2,350136250,"POSSANTES E FERREIRA, LDA",PV Lisboa,PV Lisboa
3,350136287,A.GARCIA & GARCIA - COMÉRCIO D,PV Lisboa,PV Lisboa
4,350136507,ALGARTALHOS-SUP. RESTAURAÇAO L,PV Algarve,PV Algarve
...,...,...,...,...
1042,350493164,SUPER CORADINHO,<NA>,SEM_MAPEAMENTO
1043,350494128,RENATO ZUMACH RIBEIRO-PO842282,<NA>,SEM_MAPEAMENTO
1044,350494129,RENATO ZUMACH RIBEIRO-PO842282,<NA>,SEM_MAPEAMENTO
1045,350494415,FORÇA DE VENCER RIO MOINHOS,<NA>,SEM_MAPEAMENTO


In [6]:
# ==========================
# 5. Criação das tabelas
# ==========================
registos_clientes = [
    (
        str(linha["PCODCL"]).strip(),
        "" if pd.isna(linha["PNOMCL"]) else str(linha["PNOMCL"]).strip(),
        None if pd.isna(linha["Ruta WH"]) else str(linha["Ruta WH"]).strip(),
        str(linha["Ruta Tte. Nueva"]).strip(),
        DATA_INICIO,
        None,
        FICHEIRO_DANONE.name,
    )
    for _, linha in clientes.iterrows()
]

with sqlite3.connect(DB_PATH) as con:
    con.execute("PRAGMA foreign_keys = ON")
    con.execute("PRAGMA journal_mode = WAL")

    if RECRIAR_TABELAS:
        con.execute(f'DROP TABLE IF EXISTS "{TABELA_CLIENTES}"')
        con.execute(f'DROP TABLE IF EXISTS "{TABELA_TARIFAS}"')

    con.execute(
        f'''
        CREATE TABLE IF NOT EXISTS "{TABELA_TARIFAS}" (
            tipo_local TEXT NOT NULL,
            modelo_ingresso TEXT NOT NULL,
            unidade TEXT,
            tarifa_base REAL,
            data_inicio TEXT NOT NULL,
            data_fim TEXT,
            fonte_tarifa TEXT NOT NULL,
            PRIMARY KEY (tipo_local, data_inicio)
        )
        '''
    )

    con.execute(
        f'''
        CREATE TABLE IF NOT EXISTS "{TABELA_CLIENTES}" (
            pcodcl TEXT NOT NULL,
            pnomcl TEXT NOT NULL,
            ruta_wh TEXT,
            tipo_local TEXT NOT NULL,
            data_inicio TEXT NOT NULL,
            data_fim TEXT,
            fonte TEXT NOT NULL,
            PRIMARY KEY (pcodcl, data_inicio)
        )
        '''
    )

    con.executemany(
        f'''
        INSERT INTO "{TABELA_TARIFAS}" (
            tipo_local,
            modelo_ingresso,
            unidade,
            tarifa_base,
            data_inicio,
            data_fim,
            fonte_tarifa
        )
        VALUES (?, ?, ?, ?, ?, ?, ?)
        ON CONFLICT(tipo_local, data_inicio) DO UPDATE SET
            modelo_ingresso = excluded.modelo_ingresso,
            unidade = excluded.unidade,
            tarifa_base = excluded.tarifa_base,
            data_fim = excluded.data_fim,
            fonte_tarifa = excluded.fonte_tarifa
        ''',
        registos_tarifas,
    )

    con.executemany(
        f'''
        INSERT INTO "{TABELA_CLIENTES}" (
            pcodcl,
            pnomcl,
            ruta_wh,
            tipo_local,
            data_inicio,
            data_fim,
            fonte
        )
        VALUES (?, ?, ?, ?, ?, ?, ?)
        ON CONFLICT(pcodcl, data_inicio) DO UPDATE SET
            pnomcl = excluded.pnomcl,
            ruta_wh = excluded.ruta_wh,
            tipo_local = excluded.tipo_local,
            data_fim = excluded.data_fim,
            fonte = excluded.fonte
        ''',
        registos_clientes,
    )

    con.execute(
        f'''
        CREATE INDEX IF NOT EXISTS "idx_{TABELA_CLIENTES}_tipo_local"
        ON "{TABELA_CLIENTES}" (tipo_local)
        '''
    )

print(f"✓ Tabela criada: {TABELA_TARIFAS}")
print(f"✓ Tabela criada: {TABELA_CLIENTES}")


✓ Tabela criada: Danone_Tarifas_2026
✓ Tabela criada: danone_clientes_2026


In [7]:
# ==========================
# 6. Validação final
# ==========================
with sqlite3.connect(DB_PATH) as con:
    estrutura_tarifas = pd.read_sql_query(
        f'PRAGMA table_info("{TABELA_TARIFAS}")',
        con,
    )

    resumo = pd.read_sql_query(
        f'''
        SELECT
            c.tipo_local,
            COUNT(*) AS clientes,
            t.modelo_ingresso,
            t.unidade,
            t.tarifa_base,
            t.data_inicio AS tarifa_data_inicio,
            t.data_fim AS tarifa_data_fim,
            t.fonte_tarifa
        FROM "{TABELA_CLIENTES}" AS c
        LEFT JOIN "{TABELA_TARIFAS}" AS t
          ON t.tipo_local = c.tipo_local
         AND t.data_inicio <= c.data_inicio
         AND (
                t.data_fim IS NULL
                OR t.data_fim >= c.data_inicio
             )
        WHERE c.data_inicio = ?
        GROUP BY
            c.tipo_local,
            t.modelo_ingresso,
            t.unidade,
            t.tarifa_base,
            t.data_inicio,
            t.data_fim,
            t.fonte_tarifa
        ORDER BY c.tipo_local
        ''',
        con,
        params=(DATA_INICIO,),
    )

    sem_correspondencia = pd.read_sql_query(
        f'''
        SELECT DISTINCT c.tipo_local
        FROM "{TABELA_CLIENTES}" AS c
        LEFT JOIN "{TABELA_TARIFAS}" AS t
          ON t.tipo_local = c.tipo_local
        WHERE t.tipo_local IS NULL
        ORDER BY c.tipo_local
        ''',
        con,
    )

if "combustivel_pct" in set(estrutura_tarifas["name"]):
    raise AssertionError(
        "A tabela de tarifas contém indevidamente combustivel_pct."
    )

if not sem_correspondencia.empty:
    raise AssertionError(
        "Existem clientes sem correspondência na tabela de tarifas."
    )

resumo


,tipo_local,clientes,modelo_ingresso,unidade,tarifa_base,tarifa_data_inicio,tarifa_data_fim,fonte_tarifa
0,AUCHAN sin Tte.,1,SEM_INGRESSO,ZERO,0.000000,2026-01-01,None,Sem transporte
1,Alentejo,3,POR_TONELADA,EUR_TON,157.465344,2026-01-01,None,TRANSPORTE
2,Algarve,2,POR_TONELADA,EUR_TON,81.357095,2026-01-01,None,TRANSPORTE
3,Azambuja 1,15,POR_TONELADA,EUR_TON,14.434323,2026-01-01,None,TRANSPORTE
4,Azambuja 2,2,POR_TONELADA,EUR_TON,27.556435,2026-01-01,None,TRANSPORTE
5,Azambuja 3,6,POR_TONELADA,EUR_TON,52.488448,2026-01-01,None,TRANSPORTE
6,Capilar,2,CAPILAR,None,NaN,2026-01-01,None,Capilar
7,Centro,3,POR_TONELADA,EUR_TON,38.054125,2026-01-01,None,TRANSPORTE
8,Directos Coimbra,8,POR_TONELADA,EUR_TON,167.699280,2026-01-01,None,Directos tte
9,Directos Lisboa,6,POR_TONELADA,EUR_TON,91.852620,2026-01-01,None,Directos tte
